> 对应 Lec 12。覆盖软阈值算子实现、单机 ADMM 通用框架、Lasso ADMM 和 LAD ADMM 完整代码，以及常用近端算子的代码实现。算法推导见"7. 非光滑优化与ADMM_概念解释.md"；分布式扩展见"8. 一致性优化_实现.md"。

---

## Soft-Thresholding（软阈值算子）实现

### 算法思路

软阈值算子是 L1 范数 $\lambda\|x\|_1$ 的**近端算子**（proximal operator），定义为：

$$S_\kappa(a) = \arg\min_z \left\{\lambda\|z\|_1 + \frac{1}{2\kappa^{-1}}\|z-a\|_2^2\right\} = \text{sign}(a)\cdot\max(|a|-\kappa, 0)$$

直觉：把绝对值小于阈值 $\kappa$ 的元素**直接压成 0**（这是 Lasso/LAD 产生稀疏解的根源），绝对值大于 $\kappa$ 的元素向 0 方向收缩 $\kappa$（但不改变符号）。在 ADMM 框架中，几乎所有涉及 L1 正则的子问题最终都会归结为调用这个算子。

### 向量化实现：np.sign(v) * np.maximum(np.abs(v) - κ, 0)

In [ ]:
import numpy as np

def soft_thresholding(v, kappa):
    """
    软阈值算子（逐元素操作，自动支持向量和矩阵输入）。

    Args:
        v: 输入向量或矩阵。
        kappa: 阈值（标量，必须 >= 0）。

    Returns:
        与 v 同形状的数组，每个元素独立做软阈值处理。
    """
    return np.sign(v) * np.maximum(np.abs(v) - kappa, 0.0)

### 三段推导对应代码（v>κ, v<-κ, |v|≤κ 三种情况）

把上面这一行代码拆开看，恰好对应数学定义里的三段分支：

In [ ]:
def soft_thresholding_explicit(v, kappa):
    """与 soft_thresholding 完全等价，但写成显式分支，便于理解三段逻辑。"""
    result = np.zeros_like(v)
    result[v > kappa] = v[v > kappa] - kappa      # v > kappa：正值向下收缩
    result[v < -kappa] = v[v < -kappa] + kappa    # v < -kappa：负值向上收缩
    # |v| <= kappa 的位置保持初始化的 0，无需额外处理              # 小值压到 0
    return result

向量化版本之所以等价：当 $|v|\leq\kappa$ 时 $\max(|v|-\kappa,0)=0$，乘以任何符号都是 0；当 $v>\kappa$ 时 $\text{sign}(v)=1$，结果是 $v-\kappa$；当 $v<-\kappa$ 时 $\text{sign}(v)=-1$，结果是 $-(|v|-\kappa) = -(-v-\kappa) = v+\kappa$。两种写法数值完全一致，向量化版本更快（无显式 Python 级分支与布尔索引赋值的多次扫描）。

### 输入为矩阵时的逐元素行为

`np.sign`、`np.abs`、`np.maximum` 全部是逐元素（element-wise）函数，因此 `soft_thresholding` 不需要任何修改即可直接作用于矩阵——常见于矩阵补全等场景中对每个元素独立做软阈值。注意这与"核范数软阈值"（先做 SVD，再对**奇异值**做软阈值）不同，见本文档末尾的 `prox_nuclear`。

---

## 单机 ADMM 通用三步框架

### 算法思路

ADMM 求解形如

$$\min_{x,z} f(x) + g(z) \quad \text{s.t.}\ Ax+Bz=c$$

的问题。核心思想是"变量拆分"：把一个混合了光滑项 $f$ 和非光滑项 $g$ 的难题，拆成两个各自简单的子问题交替求解，并用一个对偶变量 $u$ 协调两者，使其最终满足约束。三步迭代（缩放形式）：

### x-update：argmin_x [f(x) + ρ/2·‖Ax + Bz^k - c + u^k‖²]

### z-update：argmin_z [g(z) + ρ/2·‖Ax^{k+1} + Bz - c + u^k‖²]

### u-update：u ← u + Ax^{k+1} + Bz^{k+1} - c

In [ ]:
def admm_template(x_update_func, z_update_func, x_init, z_init, u_init,
                   A, B, c, rho, max_iter=1000, eps=1e-4, verbose=0):
    """
    单机 ADMM 三步迭代的通用骨架（x_update_func / z_update_func 由具体问题提供）。

    Args:
        x_update_func(z, u) -> 新的 x：求解 x 子问题（通常是光滑问题，闭式解或迭代法）。
        z_update_func(x, u) -> 新的 z：求解 z 子问题（通常是近端算子，如软阈值）。
        A, B, c: 约束 Ax + Bz = c 中的系数（很多场景下 A=I 或 B=-I，可以简化乘法）。
        rho: 增广拉格朗日系数。
    """
    x, z, u = x_init.copy(), z_init.copy(), u_init.copy()

    for k in range(max_iter):
        x_new = x_update_func(z, u)
        z_new = z_update_func(x_new, u)
        # u-update：把约束残差累加到对偶变量里（"记账本"，记录历史违反约束的程度）
        r = A @ x_new + B @ z_new - c
        u_new = u + r

        # 收敛判断需要原始残差和对偶残差同时达标
        primal_res = np.linalg.norm(r)
        dual_res = rho * np.linalg.norm(B.T @ (z_new - z))

        x, z, u = x_new, z_new, u_new

        if verbose and k % 1000 == 0:
            print(f"iter {k}: primal={primal_res:.6f}, dual={dual_res:.6f}")
        if primal_res < eps and dual_res < eps:
            break

    return k + 1, x, z, u

### 原始残差 r 与对偶残差 s 的计算（代码技巧：r = u_new - u_old）

**关键代码技巧**：从 $u$ 更新公式 $u^{k+1} = u^k + r^{k+1}$ 可以反推出 $r^{k+1} = u^{k+1} - u^k$。也就是说，原始残差不需要重新计算 $Ax+Bz-c$，直接用更新前后的 $u$ 差值即可：

In [ ]:
# 两种写法完全等价，但第二种省一次矩阵乘法
# 写法一（显式重算约束残差）
r = A @ x_new + B @ z_new - c

# 写法二（利用 u 更新公式反推，更高效）
r = u_new - u_old

### 双停止条件判断（primal_res < eps AND dual_res < eps）

两个残差物理意义不同，**必须同时**满足才能停止：

- **原始残差**衡量约束 $Ax+Bz=c$ 是否被满足（即 $x$ 和 $z$ 是否已经"对齐"）。
- **对偶残差**衡量 $z$ 是否已经稳定不再变化（即迭代是否已经收敛，不只是恰好这一步约束满足）。

只看原始残差可能在 $z$ 仍大幅波动时误判为收敛；只看对偶残差可能在约束严重违反时误判为收敛。

---

## 单机 Lasso ADMM 完整实现

### 算法思路

### 问题形式：min ½‖y-Xβ‖² + λ‖z‖₁，s.t. β=z

引入辅助变量 $z$ 把光滑的最小二乘项和非光滑的 L1 惩罚拆开：$\beta$ 负责拟合数据（光滑子问题），$z$ 负责处理稀疏惩罚（非光滑子问题），约束 $\beta=z$ 保证两者最终一致。对应 ADMM 标准形式：$f(\beta)=\frac12\|y-X\beta\|^2$，$g(z)=\lambda\|z\|_1$，$A=I,B=-I,c=0$。

### β-update：(X^TX + ρI)^{-1}(X^Ty + ρ(z^k - u^k))——Cholesky 预分解

对 $\beta$ 子问题求导令其为零，得到一个形似**岭回归**正规方程的闭式解：

$$\beta^{k+1} = (X^TX+\rho I)^{-1}\left(X^Ty+\rho(z^k-u^k)\right)$$

由于 $(X^TX+\rho I)$ 在整个迭代过程中**不变**（$\rho$ 固定），可以在循环外预先做一次 Cholesky 分解，循环内只需做代价更低的回代求解。

### Cholesky 预分解优化（cho_factor 只做一次，cho_solve 每轮用）

In [ ]:
import numpy as np
from scipy.linalg import cho_factor, cho_solve

def admm_lasso(X, y, lam, rho=1.0, maxit=10000, eps=1e-3, verbose=0):
    """
    单机 Lasso ADMM：min 1/2||y-X*beta||^2 + lam*||beta||_1

    Args:
        X: 设计矩阵 (n, p)。
        y: 响应向量 (n,)。
        lam: L1 正则化系数。
        rho: 增广拉格朗日系数（影响收敛速度，不影响最终解）。
        maxit: 最大迭代次数。
        eps: 残差收敛阈值。
        verbose: >0 时每 1000 步打印残差。

    Returns:
        (实际迭代次数, beta 估计值)。
    """
    n, p = X.shape
    beta = np.zeros(p)
    z = np.zeros(p)     # 辅助变量，维度与 beta 相同（p 维）
    u = np.zeros(p)     # 对偶变量，维度与 z 相同

    # 关键优化：Cholesky 分解只做一次，放在循环外面
    A_cho = cho_factor(X.T @ X + rho * np.eye(p))
    Xty = X.T @ y        # X^T y 也只需要算一次，循环内重复用

    for i in range(maxit):
        # Step 1: beta 更新（Cholesky 回代，O(p^2) 而非重新分解的 O(p^3))
        rhs = Xty + rho * (z - u)
        beta_new = cho_solve(A_cho, rhs)

        # Step 2: z 更新（软阈值，阈值是 lam/rho）
        z_old = z
        z = soft_thresholding(beta_new + u, lam / rho)

        # Step 3: u 更新（廉价的向量加法）
        u = u + beta_new - z

        # 收敛判断
        resid_r = np.linalg.norm(beta_new - z)          # 原始残差：beta 和 z 的差距
        resid_s = rho * np.linalg.norm(z - z_old)        # 对偶残差：z 本身的变化量

        beta = beta_new

        if verbose > 0 and i % 1000 == 0:
            print(f"Iter {i}: ||r||={resid_r:.6f}, ||s||={resid_s:.6f}")

        if resid_r <= eps and resid_s <= eps:
            break

    return i + 1, beta

### z-update：S_{λ/ρ}(β + u)——软阈值

注意这里软阈值的输入是 `beta_new + u`（已经更新到本轮的 $\beta$），不是上一轮的 `beta`——ADMM 三步迭代中，后面的子问题总是用**最新**算出来的变量。

### u-update 与收敛判断完整代码

见上方完整实现的 Step 3 和收敛判断部分。

### ρ 参数效果：ρ 过大/过小的收敛曲线表现

| $\rho$ | 现象 |
|---|---|
| 太小 | $\beta$-update 子问题里"贴近 $z$"的惩罚太弱，原始残差（$\|\beta-z\|$）下降很慢 |
| 太大 | 子问题过度强调贴近 $z$，可能导致 $z$（从而对偶残差）震荡 |

实践中常见做法：先取 $\rho=1$ 跑一遍，观察两个残差的下降曲线，如果原始残差降得明显比对偶残差慢就调大 $\rho$，反之调小。$\rho$ 只影响收敛速度，不影响最终收敛到的解。

---

## 单机 LAD ADMM 完整实现

### 算法思路

### 问题形式：min ‖y-Xβ‖₁，s.t. β=θ（辅助变量 θ）

LAD（Least Absolute Deviation，最小绝对偏差回归）用 L1 损失代替平方损失，对异常值更稳健。这里非光滑项是损失本身（不是正则项），所以拆分方式略有不同：令 $z = X\beta - y$（残差），把 $\|\cdot\|_1$ 整个放到 $z$ 上：

$$\min_{\beta,z}\|z\|_1 \quad\text{s.t.}\quad X\beta-y-z=0$$

对应 $f(\beta)=0$（无惩罚项！），$g(z)=\|z\|_1$，$A=X, B=-I, c=y$。

### β-update：OLS 子问题（X^TX 正规方程，Cholesky 求解）

因为 $f(\beta)=0$，$\beta$ 子问题只剩二次惩罚项，令梯度为零得到一个**无正则的**最小二乘正规方程：

$$\beta^{k+1} = (X^TX)^{-1}X^T(y+z^k-u^k)$$

注意这里系数矩阵是 $X^TX$（没有 $+\rho I$），因为 LAD 的约束形式里 $\beta$ 没有被直接拉向 $z$，而是通过 $X\beta$ 间接关联。

### z-update：S_{1/ρ}(Xβ^{k+1} + u - y) + y——目标空间软阈值

In [ ]:
def admm_lad(A, b, rho=1.0, max_iter=10000, eps=1e-3, verbose=0):
    """
    单机 LAD ADMM：min ||A*x - b||_1

    与 Lasso ADMM 的关键区别：
    - z, u 的维度是 n（残差维度），而不是 p（系数维度）
    - beta-update 没有 +rho*I 正则项（因为 f(beta)=0）
    - z-update 的软阈值阈值是 1/rho，不是 lam/rho
    """
    n, p = A.shape
    x = np.zeros(p)     # 回归系数 beta
    z = np.zeros(n)     # 辅助变量，维度是 n（残差！）
    u = np.zeros(n)     # 对偶变量，维度同样是 n

    # Cholesky 分解只做一次：注意这里分解的是 A^T A，没有 rho 项
    A_cho = cho_factor(A.T @ A)

    for i in range(max_iter):
        # Step 1: x（beta）更新——无正则的最小二乘
        xnew = cho_solve(A_cho, A.T @ (b + z - u))

        # Step 2: z 更新——在"目标残差空间"做软阈值，注意 +u 和 -b 的顺序
        Axnew = A @ xnew                                  # 缓存 Ax，避免重复计算
        znew = soft_thresholding(Axnew - b + u, 1.0 / rho)

        # Step 3: u 更新
        unew = u + Axnew - znew - b

        # 收敛判断：原始残差可以直接用 u 的差值（同 Lasso 版本的技巧）
        resid_r = np.linalg.norm(unew - u)
        resid_s = rho * np.linalg.norm(A.T @ (znew - z))

        x, z, u = xnew, znew, unew

        if verbose > 0 and i % 1000 == 0:
            print(f"Iter {i}: ||r||={resid_r:.6f}, ||s||={resid_s:.6f}")

        if resid_r <= eps and resid_s <= eps:
            break

    return i + 1, x

### LAD vs Lasso ADMM 的z-update 区别对比

| 对比项 | LAD | Lasso |
|---|---|---|
| 目标函数 | $\|X\beta-y\|_1$ | $\frac12\|y-X\beta\|_2^2+\lambda\|\beta\|_1$ |
| $f(\beta)$ | 0（无惩罚） | $\frac12\|y-X\beta\|_2^2$ |
| $\beta$-update 系数矩阵 | $X^TX$（无 $\rho I$） | $X^TX+\rho I$（类似岭回归） |
| $z,u$ 的维度 | $n$（残差） | $p$（系数） |
| 软阈值阈值 | $1/\rho$ | $\lambda/\rho$ |
| 软阈值作用对象 | $X\beta-y+u$（目标空间残差） | $\beta+u$（参数空间） |

**最容易混淆的两点**：(1) LAD 的 $z,u$ 是 $n$ 维而 Lasso 是 $p$ 维；(2) LAD 没有正则化参数 $\lambda$，软阈值阈值直接是 $1/\rho$。

---

## 近端算子通用代码

### 算法思路

ADMM 的 $z$-update 在很多场景下都能化简为某个正则项/约束的**近端算子**：

$$\text{prox}_{\lambda h}(v) = \arg\min_z \left\{h(z)+\frac{1}{2\lambda}\|z-v\|_2^2\right\}$$

不同的 $h$（正则项或约束的指示函数）对应不同的、通常有**闭式解**的近端算子，下面是几个常考的实现。

### prox_l1（L1 正则，软阈值）

In [ ]:
def prox_l1(v, lam_over_rho):
    """L1 正则的近端算子，就是软阈值算子（与上面 soft_thresholding 完全相同）。"""
    return soft_thresholding(v, lam_over_rho)

### prox_nonneg（非负约束，逐元素 max(v, 0)）

**思路解释**：非负约束 $x\geq 0$ 可以写成指示函数 $h(x) = I_{x\geq0}(x)$（在约束集合内取 0，否则取 $+\infty$）。其近端算子是把每个分量截断到非负区间——这恰好是 ReLU 函数：

In [ ]:
def prox_nonneg(v):
    """非负约束的近端算子：逐元素截断到 >= 0。"""
    return np.maximum(v, 0.0)

### prox_box（box 约束 [a,b]，逐元素 clip）

In [ ]:
def prox_box(v, a, b):
    """box 约束 a <= x <= b 的近端算子：逐元素裁剪到区间 [a, b]。"""
    return np.clip(v, a, b)

### prox_nuclear（矩阵核范数，SVD 后对奇异值软阈值）

**思路解释**：核范数 $\|X\|_* = \sum_i \sigma_i(X)$（奇异值之和）是矩阵补全/低秩恢复问题里常用的"矩阵版 L1 范数"——它对奇异值施加稀疏惩罚，鼓励矩阵是低秩的（即只有少数几个奇异值非零）。其近端算子是：先做 SVD 分解，对**奇异值向量**做软阈值（不是对矩阵的每个元素直接做软阈值），再用阈值化后的奇异值重新合成矩阵。

In [ ]:
def prox_nuclear(V, kappa):
    """
    核范数的近端算子：SVD 分解后对奇异值做软阈值，再重新合成矩阵。

    Args:
        V: 输入矩阵 (m, n)。
        kappa: 阈值。
    """
    U, sigma, Vt = np.linalg.svd(V, full_matrices=False)
    sigma_thresh = soft_thresholding(sigma, kappa)   # 只对奇异值（向量）做软阈值
    return U @ np.diag(sigma_thresh) @ Vt

**与 `prox_l1` 的关键区别**：`prox_l1` 直接对输入的每个元素做软阈值；`prox_nuclear` 必须先转换到"奇异值空间"才能做软阈值，因为核范数惩罚的对象是奇异值，不是矩阵元素本身。这是矩阵范数近端算子和向量范数近端算子在实现上的本质差异。